# 규칙 baseline (사업보고서 정정요구 사전 점검)

1단계 라벨 표 불러오기 → 2단계 정정 전·후 본문 가져오기 (OpenDART 공식 API) → 3단계 규칙으로 판정하고 recall 계산 → 4단계 놓친 것 보기

받은 보고서 원문은 `dart_xml/`에, 본문 붙인 표는 `labels_text.tsv`에 저장됨. 규칙만 바꿔 다시 돌릴 땐 3단계부터

In [2]:
KEY = ""   # 본인 OpenDART 인증키. 깃허브에 올릴 땐 반드시 다시 비울 것
assert len(KEY) == 40, "키 40자리를 넣어주세요"

import pandas as pd

## 1. 라벨 표 불러오기
금감원 요구로 고친 기록. `req`가 여/예인 행 = 금감원이 지적한 것

In [3]:
URL = "https://raw.githubusercontent.com/hyemin-hub/annual-report-precheck/main/data/labels/labels_2025_06-08.tsv"
df = pd.read_csv(URL, sep="\t", dtype=str).fillna("")
df = df.rename(columns={"rcp": "rcp_no", "corp": "corp_name"})
df = df[df["req"].isin(["여", "예"])].reset_index(drop=True)
print(len(df), "행 (금감원 지적),", df["rcp_no"].nunique(), "개 보고서")
df.head()

122 행 (금감원 지적), 101 개 보고서


,rcp_no,corp_name,req,item,reason
0,20250829000681,신영증권,예,Ⅰ. 회사의 개요 4. 주식의 총수 등 라. 자기주식에 관한 사항,기재 미흡
1,20250828000839,노드메이슨,예,Ⅲ. 재무에 관한 사항1. 요약재무정보가. 연결재무정보,정정기재
2,20250828000839,노드메이슨,예,Ⅲ. 재무에 관한 사항1. 요약재무정보나. 별도재무정보,정정기재
3,20250828000839,노드메이슨,예,Ⅲ. 재무에 관한 사항6. 배당에 관한 사항,정정기재
4,20250828000839,노드메이슨,예,Ⅲ. 재무에 관한 사항8. 기타 재무에 관한 사항,정정기재


## 2. 정정 전·후 본문 가져오기
- 정정신고서 원문을 받음 → 그 안에 회사 고유번호가 있음
- 고유번호로 같은 회사의 원본 사업보고서를 찾아 원문을 받음
- 항목(예: `Ⅰ. 회사의 개요 4. 주식의 총수 등 라. 자기주식에 관한 사항`)에 해당하는 부분을 두 원문에서 잘라 `before`(원본), `after`(정정본)에 넣음

`status`: ok_sub = 소제목(라. 등)까지 잘라냄 / ok = 번호 섹션(4. 등) 전체 / ok_top = 큰 섹션(VI. 등) 전체만 찾음, 너무 넓음 / no_section = 섹션 못 찾음 / no_orig = 원본 보고서 못 찾음

섹션은 항목의 번호(VI. 3. 나.)로 찾음. 회사마다 제목 표기가 달라도('3. 주주총회 등' vs '3. 주주총회 등에 관한 사항') 번호는 같음

처음엔 `N_TEST = 10`으로 확인. 괜찮으면 `None`으로 바꿔 전체 실행 (보고서 약 200개 다운로드, 10분 안팎)

In [4]:
import io, os, re, time, zipfile, datetime, requests
from bs4 import BeautifulSoup, XMLParsedAsHTMLWarning
import warnings
warnings.filterwarnings("ignore", category=XMLParsedAsHTMLWarning)

DOC = "https://opendart.fss.or.kr/api/document.xml"
LIST = "https://opendart.fss.or.kr/api/list.json"
CACHE = "dart_xml"
MAX_CHARS = 30000
ROMAN = {"Ⅰ": "I", "Ⅱ": "II", "Ⅲ": "III", "Ⅳ": "IV", "Ⅴ": "V", "Ⅵ": "VI", "Ⅶ": "VII",
         "Ⅷ": "VIII", "Ⅸ": "IX", "Ⅹ": "X", "Ⅺ": "XI", "Ⅻ": "XII"}
_parsed = {}


def get_xml(rcp_no, key):
    """공시서류원본파일 API로 보고서 XML을 받아 dart_xml/에 저장. 이미 있으면 다시 안 받음."""
    os.makedirs(CACHE, exist_ok=True)
    path = os.path.join(CACHE, f"{rcp_no}.xml")
    if not os.path.exists(path):
        r = requests.get(DOC, params=dict(crtfc_key=key, rcept_no=rcp_no), timeout=60)
        if r.content[:2] != b"PK":  # zip이 아니면 오류 메시지(XML)가 온 것
            raise RuntimeError("document.xml 오류: " + r.text[:200])
        z = zipfile.ZipFile(io.BytesIO(r.content))
        name = f"{rcp_no}.xml" if f"{rcp_no}.xml" in z.namelist() else max(z.namelist(), key=lambda n: z.getinfo(n).file_size)
        open(path, "wb").write(z.read(name))
        time.sleep(0.5)
    return open(path, encoding="utf-8", errors="ignore").read()


def to_ascii(s):
    s = str(s)
    for k, v in ROMAN.items():
        s = s.replace(k, v)
    return s


RE_ROMAN = re.compile(r"\s*(XII|XI|X|IX|VIII|VII|VI|V|IV|III|II|I)\s*\.")


def norm(s):
    s = str(s)
    for k, v in ROMAN.items():
        s = s.replace(k, v)
    return re.sub(r"[\s\.\,ㆍ·\(\)【】]", "", s)


def parse(rcp_no, key):
    """→ (corp_code, [(제목, 본문), ...])  SECTION-1/2/3 단위"""
    if rcp_no in _parsed:
        return _parsed[rcp_no]
    xml = get_xml(rcp_no, key)
    m = re.search(r'<COMPANY-NAME AREGCIK="(\d{8})"', xml)
    corp_code = m.group(1) if m else None
    soup = BeautifulSoup(xml, "lxml")
    secs, roman = [], None
    for sec in soup.find_all(re.compile(r"^section-\d$")):
        t = sec.find("title")
        if not t:
            continue
        title = t.get_text(" ", strip=True)
        level = int(sec.name[-1])
        if level == 1:
            m = RE_ROMAN.match(to_ascii(title))
            roman = m.group(1) if m else None
            num = None
        else:
            m = re.match(r"\s*(\d{1,2})\s*\.(?!\d)", title)
            num = m.group(1) if m else None
        text = re.sub(r"[ \t\u3000\xa0]+", " ", sec.get_text("\n", strip=True))
        secs.append(dict(title=title, text=re.sub(r"\n{2,}", "\n", text), level=level, roman=roman, num=num))
    _parsed[rcp_no] = (corp_code, secs)
    return _parsed[rcp_no]


def find_original(corp_code, rcept_dt, key, exclude_rcp=None):
    """같은 회사, 정정신고 접수일 이전 400일 안에서 이번 정정신고서를 뺀 가장 최근 사업보고서 접수번호."""
    end = datetime.datetime.strptime(rcept_dt, "%Y%m%d")
    bgn = (end - datetime.timedelta(days=400)).strftime("%Y%m%d")
    r = requests.get(LIST, params=dict(crtfc_key=key, corp_code=corp_code, bgn_de=bgn, end_de=rcept_dt,
                                       pblntf_ty="A", last_reprt_at="N", page_count=100), timeout=30).json()
    if r.get("status") not in ("000", "013"):
        raise RuntimeError(f"list.json {r.get('status')} {r.get('message')}")
    c = [x for x in r.get("list", []) if "사업보고서" in x["report_nm"]
         and x["rcept_no"] != exclude_rcp]  # 이전에 이미 정정한 적 있으면 직전 보고서가 [기재정정]일 수 있음 → 이름 안 가림
    c.sort(key=lambda x: x["rcept_no"], reverse=True)
    return c[0]["rcept_no"] if c else None


def parse_item(item):
    """'VI. 이사회 등 ... 3. 주주총회 등 나. 소수주주권의 행사 여부' → ('VI', '3', '나')"""
    it = to_ascii(item)
    m = RE_ROMAN.match(it)
    roman = m.group(1) if m else None
    rest = it[m.end():] if m else it
    n = re.search(r"(?<![\d\-])(\d{1,2})\s*\.(?!\d)", rest)
    num = n.group(1) if n else None
    rest2 = rest[n.end():] if n else rest
    l = re.search(r"([가나다라마바사아자차카타파하])\s*\.", rest2)
    head = norm(rest2[l.end():])[:4] if l else ""
    return roman, num, (l.group(1) if l else None), head


SUB = re.compile(r"^\s*([가나다라마바사아자차카타파하])\s*\.\s*")


def sub_slice(letter, head, text):
    """본문에서 '나. (항목과 같은 제목)'으로 시작하는 줄 → 다음 '가.~하.' 줄 직전까지.
    글자만 같고 제목이 다르면(회사마다 소제목 체계가 다름) 자르지 않음"""
    lines = text.split("\n")
    for i, ln in enumerate(lines):
        m = SUB.match(ln)
        if m and m.group(1) == letter and head and head in norm(ln):
            for j in range(i + 1, len(lines)):
                if SUB.match(lines[j]):
                    return "\n".join(lines[i:j])
            return "\n".join(lines[i:])
    return None


def section_text(rcp_no, item, key):
    """→ (본문, 상태)
    ok_sub: 소제목(나. 등)까지 잘라냄 / ok: 번호 섹션(3. 등) 전체 / ok_top: 큰 섹션(VI. 등) 전체만 찾음(너무 넓음)
    no_section: 못 찾음"""
    _, secs = parse(rcp_no, key)
    roman, num, letter, head = parse_item(item)
    hit = None
    if roman and num:
        hit = next((s for s in secs if s["level"] == 2 and s["roman"] == roman and s["num"] == num), None)
    if hit is None and roman and not num:
        hit = next((s for s in secs if s["level"] == 1 and s["roman"] == roman), None)
        if hit:
            return hit["text"][:MAX_CHARS], "ok"
    if hit is None:  # 번호 체계가 이상한 항목: 제목 글자로 찾기
        it = norm(item)
        cands = [s for s in secs if len(norm(s["title"])) >= 4 and norm(s["title"]) in it]
        if cands:
            hit = max(cands, key=lambda s: (s["level"], len(s["title"])))
            if hit["level"] == 1 and num:
                return hit["text"][:MAX_CHARS], "ok_top"
    if hit is None:
        return "", "no_section"
    if letter:
        part = sub_slice(letter, head, hit["text"])
        if part:
            return part[:MAX_CHARS], "ok_sub"
    return hit["text"][:MAX_CHARS], "ok"


In [6]:
N_TEST = None      # 전체 돌릴 땐 None
work = (df if N_TEST is None else df.head(N_TEST)).copy()
for c in ["orig_rcp_no", "before", "after", "status"]:
    work[c] = ""

for i, row in work.iterrows():
    rcp = row["rcp_no"]
    try:
        corp_code, _ = parse(rcp, KEY)
        orig = find_original(corp_code, rcp[:8], KEY, exclude_rcp=rcp)
        work.at[i, "orig_rcp_no"] = orig or ""
        after, st_a = section_text(rcp, row["item"], KEY)
        work.at[i, "after"] = after
        if orig:
            before, st_b = section_text(orig, row["item"], KEY)
            work.at[i, "before"] = before
            status = st_b
        else:
            status = "no_orig"
    except Exception as e:
        status = f"err: {str(e)[:80]}"
    work.at[i, "status"] = status
    print(i + 1, row["corp_name"], "|", status, "| 정정 전", len(work.at[i, "before"]), "자 / 정정 후", len(work.at[i, "after"]), "자")

work.to_csv("labels_text.tsv", sep="\t", index=False)
print(work["status"].value_counts())

1 신영증권 | ok_sub | 정정 전 7011 자 / 정정 후 7222 자
2 노드메이슨 | ok | 정정 전 1819 자 / 정정 후 1798 자
3 노드메이슨 | ok | 정정 전 1819 자 / 정정 후 1798 자
4 노드메이슨 | ok | 정정 전 1755 자 / 정정 후 1749 자
5 노드메이슨 | ok | 정정 전 3655 자 / 정정 후 3908 자
6 노드메이슨 | ok | 정정 전 4058 자 / 정정 후 4052 자
7 파인디지털 | ok | 정정 전 2039 자 / 정정 후 2298 자
8 에이디테크놀로지 | ok | 정정 전 46 자 / 정정 후 1687 자
9 세토피아 | ok_sub | 정정 전 1305 자 / 정정 후 1301 자
10 세토피아 | ok_sub | 정정 전 1220 자 / 정정 후 2442 자
11 아이큐어 | ok | 정정 전 2888 자 / 정정 후 4418 자
12 비유테크놀러지 | ok | 정정 전 2533 자 / 정정 후 3757 자
13 콜마홀딩스 | ok_sub | 정정 전 18 자 / 정정 후 163 자
14 콜마홀딩스 | ok_sub | 정정 전 1225 자 / 정정 후 1615 자
15 대웅제약 | ok | 정정 전 2314 자 / 정정 후 861 자
16 대웅 | ok_sub | 정정 전 2118 자 / 정정 후 2919 자
17 한진칼 | ok | 정정 전 5609 자 / 정정 후 6195 자
18 삼성바이오로직스 | ok | 정정 전 5624 자 / 정정 후 5855 자
19 제일바이오 | ok_sub | 정정 전 651 자 / 정정 후 1269 자
20 제일바이오 | ok_sub | 정정 전 1274 자 / 정정 후 1123 자
21 파멥신 | ok | 정정 전 465 자 / 정정 후 1275 자
22 파멥신 | ok | 정정 전 465 자 / 정정 후 1275 자
23 하나기술 | ok | 정정 전 2370 자 / 정정 후 4447 자
24 KG케미칼 | ok_sub | 정정 전 42

## 3. 규칙 baseline
`before` 글만 보고 판정: 비었거나 / '해당사항 없음'이거나 / 섹션별 최소 글자수보다 짧으면 → "지적받겠다"로 찍음.
정답이 전부 '지적받은 것'이므로 찍은 비율이 recall

In [12]:
import re
EMPTY = re.compile(r"해당\s*사항\s*없음|해당없음|^\s*-\s*$|^\s*$")
MIN_CHARS = {"자기주식": 150, "소수주주권": 100, "공급계약": 150, "재무": 100, "기타": 100}


def section_of(item: str) -> str:
    if re.search(r"XI|공시내용|단일판매|공급계약", item): return "공급계약"
    if re.search(r"VI|Ⅵ|주주총회|소수주주|의사록", item): return "소수주주권"
    if re.search(r"자기주식|주식의 총수", item): return "자기주식"
    if re.search(r"III|Ⅲ|재무", item): return "재무"
    return "기타"


def rule_flag(before: str, item: str, reason: str = "") -> bool:
    sec = section_of(item)
    if isinstance(before, str) and before.strip():
        t = before.strip()
        return bool(EMPTY.search(t)) or len(t) < MIN_CHARS[sec] or "미첨부" in reason
    # before 텍스트가 없으면 사유 키워드로 근사 (1주차 수작업 분류와 동일 기준)
    return bool(re.search(r"미기재|누락|미첨부|미작성|신규작성", reason))




In [13]:
t = pd.read_csv("labels_text.tsv", sep="\t", dtype=str).fillna("")
t = t[t["before"].str.strip() != ""].copy()
t["section"] = t["item"].map(section_of)
t["flag"] = [rule_flag(b, it, r) for b, it, r in zip(t["before"], t["item"], t["reason"])]
print(f"본문 채워진 {len(t)}행 기준 규칙 recall")
for sec, g in t.groupby("section"):
    print(f"  {sec}: {g.flag.sum()}/{len(g)} = {g.flag.mean():.0%}")
print(f"  전체: {t.flag.sum()}/{len(t)} = {t.flag.mean():.0%}")

본문 채워진 114행 기준 규칙 recall
  공급계약: 15/48 = 31%
  기타: 0/1 = 0%
  소수주주권: 14/41 = 34%
  자기주식: 6/20 = 30%
  재무: 0/4 = 0%
  전체: 35/114 = 31%


## 4. 놓친 것 들여다보기
규칙이 못 잡은 행의 원본 글과 고친 글. 여기서 '왜 놓쳤나'를 1건 골라 적어오기

In [14]:
miss = t[~t.flag]
for _, r in miss.head(3).iterrows():
    print("■", r["corp_name"], "|", r["item"], "| 사유:", r["reason"])
    print("  정정 전:", r["before"][:300].replace("\n", " "))
    print("  정정 후:", r["after"][:300].replace("\n", " "), "\n")

■ 신영증권 | Ⅰ. 회사의 개요 4. 주식의 총수 등 라. 자기주식에 관한 사항 | 사유: 기재 미흡
  정정 전: 라. 자기주식에 관한 사항 (1) 보유현황 (단위 : 주) 취득방법 취득(계약체결)결정 공시일자 주식종류 취득기간 취득목적 최초보유예상기간 최초취득수량 변동 수량 기말수량 비고 처분(-) 소각(-) 배당가능이익범위이내취득 직접취득 1994.09.26 우선주 1994.10.04~1995.03.11 - - 658,000 - - 658,000 - 1995.04.24 우선주 1995.05.11~1995.06.11 - - 64,000 - - 722,000 - 1995.04.24 보통주 1995.05.06~1995.06.18 - - 100,0
  정정 후: 라. 자기주식에 관한 사항(1) 보유현황 (단위 : 주) 취득방법 취득(계약체결)결정 공시일자 주식종류 취득기간 취득목적 최초보유예상기간 최초취득수량 변동 수량 기말수량 비고 처분(-) 소각(-) 배당가능이익범위이내취득 직접취득 1995.04.24 보통주 1995.05.06~1995.06.18 - - 100,000 100,000 - - 주1) 1996.06.13 보통주 1996.06.19~1996.07.26 - - 100,000 100,000 - - 주2) 1997.03.27 보통주 1997.04.02~1997.07.02 증시의 가 

■ 노드메이슨 | Ⅲ. 재무에 관한 사항1. 요약재무정보가. 연결재무정보 | 사유: 정정기재
  정정 전: 1. 요약재무정보 가. 요약 재무정보 - 연결 (단위 : 백만원) 사업연도 구분 2024연도 (제12기) 2023연도 (제11기) 2022연도 (제10기) 회계처리 기준 K-IFRS K-IFRS K-IFRS 감사인(감사의견) 적정 적정 적정 유동자산 매출채권 재고자산 비유동자산 투자자산 유형자산 무형자산 기타비유동자산 자산총계 7,0062,0504,10817,397012,8087353,85424,403 7,8112,5834,09815,24422311,0738553,

## 5. 결과 기록
위 recall을 `eval/results.csv` 형식 한 줄로 출력. 이 줄을 저장소의 `eval/results.csv`에 붙여 넣고 PR

In [15]:
import datetime
rec = {s: f"{g.flag.mean():.0%}" for s, g in t.groupby("section")}
# 열 순서: date,method,recall_자기주식,recall_소수주주권,recall_공급계약,recall_재무,fp_per100,evidence_overlap,n_pos,n_neg,note
line = [datetime.date.today().isoformat(), "규칙(빈칸·해당없음·글자수)",
        rec.get("자기주식", ""), rec.get("소수주주권", ""), rec.get("공급계약", ""), rec.get("재무", ""),
        "", "", str(len(t)), "", f"전체 {t.flag.mean():.0%}. before 채워진 행만"]
print(",".join(line))

2026-10-07,규칙(빈칸·해당없음·글자수),30%,34%,31%,0%,,,114,,전체 31%. before 채워진 행만


## 6. 정정 유형 분포
정정 전 글자 수와 전후 길이 비율로 지적 유형을 나눔. 빈칸은 규칙이 잡는 몫, 나머지는 LLM이 맡을 몫

In [16]:
def kind(b, a):
    lb, la = len(b), len(a)
    if lb < 100: return "빈칸·한 줄 (100자 미만)"
    r = la / max(lb, 1)
    if r >= 1.5: return "부연 추가 (1.5배 이상)"
    if r >= 1.1: return "소폭 추가 (1.1~1.5배)"
    if r >= 0.9: return "길이 유지 (숫자·문구 교체)"
    return "줄어듦"
t["kind"] = [kind(b, a) for b, a in zip(t["before"], t["after"])]
print(t["kind"].value_counts().to_string())
print()
print("유형별 규칙 recall")
print(t.groupby("kind")["flag"].mean().map(lambda x: f"{x:.0%}").to_string())

kind
부연 추가 (1.5배 이상)     39
소폭 추가 (1.1~1.5배)    31
길이 유지 (숫자·문구 교체)    22
빈칸·한 줄 (100자 미만)    18
줄어듦                  4

유형별 규칙 recall
kind
길이 유지 (숫자·문구 교체)     14%
부연 추가 (1.5배 이상)       3%
빈칸·한 줄 (100자 미만)    100%
소폭 추가 (1.1~1.5배)     42%
줄어듦                   0%
